In [ ]:
 self._notify('game_over', mines_gird=self.mines_grid)
            redraw_board(canvas)
            output_area.clear_output()
            show_game_over_message()

_notify('win')
            redraw_board(canvas)
            show_win_message()
              self.flag_all_mines()
            return

self._notify('reveal')
        redraw_board(canvas) 
    reveal_all_mines()

self._notify('new_game, size=.self.size, n_mines=self.n_mines)
        # Log-Bereich einmal löschen bei neuem Spiel
        output_area.clear_output()
        log('Neues Spiel gestartet.')
        log(f'Spielfeld: {GRID_SIZE}x{GRID_SIZE}, Minen: {NUMBER_OF_MINES}')
        log(f'Flaggen-Modus: {'aktiv' if flag_mode_button.value else 'inaktiv'}')
             redraw_board(canvas)    

self._notify('flag', status=self.flag_grid[row][col])
        status = 'gesetzt' if flag_grid[row][col] else 'entfernt'
        log(f'Flagge {status} auf Feld ({row}, {col}).')
        redraw_board(canvas)

In [ ]:
def redraw_board(canvas):
    '''
    Zeichnet das komplette Spielfeld neu über das Darstellungsmodul.
    '''
    D.update(
        canvas,
        BOARD_SPEC,
        event='redraw',
        mines_grid=mines_grid,
        visibility_grid=visibility_grid,
        flag_grid=flag_grid,
        neighbor_mine_counts=neighbor_mine_counts,
    )

In [ ]:
import random
import traceback


from ipycanvas import Canvas
from ipywidgets import VBox, HBox, Button, ToggleButton, Output
from IPython.display import display

import darstellung as D

# --------------------------------------------------
# Konfiguration des Spiels
# --------------------------------------------------

GRID_SIZE = 10       # Anzahl Zeilen und Spalten des Spielfelds
NUMBER_OF_MINES = 10    # Anzahl Minen im Spielfeld

CELL_SIZE = 20          # Pixelfläche eines Feldes (Breite und Höhe)

BOARD_SPEC = (CELL_SIZE, CELL_SIZE, CELL_SIZE, CELL_SIZE, GRID_SIZE, GRID_SIZE)


output_area = Output()

# Toggle-Button für Flaggenmodus
flag_mode_button = ToggleButton(
    description='Flaggen-Modus',
    value=False,
    tooltip='Aktiv: Klick = Flagge setzen/entfernen\nInaktiv: Klick = Feld aufdecken',
)

In [ ]:
@output_area.capture(clear_output=False)
def handle_mouse_click(x, y, canvas):
    '''
    Reagiert auf Mausklicks im Spielfeldbereich.

    Steuerung:
    - Linksklick (Flaggen-Modus AUS)  -> Feld aufdecken
    - Linksklick (Flaggen-Modus EIN)  -> Flagge setzen/entfernen

    Args:
        x (float): x-Koordinate im Canvas.
        y (float): y-Koordinate im Canvas.
    '''
    x0, y0, dx, dy, ncol, nrow = BOARD_SPEC
    log(f'Klick bei Canvas-Koordinate ({x:.1f}, {y:.1f})')
    log(f'Flaggen-Modus aktuell: {'aktiv' if flag_mode_button.value else 'inaktiv'}')

    # Prüfen, ob der Klick innerhalb des Spielfeldbereichs liegt
    if not (x0 <= x < x0 + ncol * dx and y0 <= y < y0 + nrow * dy):
        log('Klick ausserhalb des Spielfeldes – ignoriert.')
        return

    col = int((x - x0) // dx)
    row = int((y - y0) // dy)
    log(f'Berechnetes Feld: row={row}, col={col}')

    try:
        if flag_mode_button.value:
            toggle_flag(row, col, canvas)
        else:
            reveal_cell(row, col, canvas)
    except Exception:
        print('FEHLER IM CLICK-HANDLER:')
        traceback.print_exc()


@output_area.capture(clear_output=False)
def on_flag_mode_change(change):
    '''
    Callback, wenn der Flaggen-Modus-Button umgeschaltet wird.

    Args:
        change (dict): Änderungsinformation von ipywidgets.
    '''
    if change['name'] == 'value':
        status = 'aktiv' if change['new'] else 'inaktiv'
        log(f'Flaggen-Modus umgeschaltet: {status}')


@output_area.capture(clear_output=False)
def new_game_clicked(button, canvas):
    '''
    Event-Handler für den 'Neues Spiel'-Button.

    Args:
        button: Das Button-Objekt, das das Event ausgelöst hat.
    '''
    log("Button 'Neues Spiel' geklickt.")
    initialize_game(canvas)


# --------------------------------------------------
# Startfunktion
# --------------------------------------------------

def start_game(grid_size=10, nmines=10):
    '''
    Startet das Minesweeper-Spiel im Jupyter-Notebook.

    - initialisiert das Spielfeld
    - verbindet Maus-Events und Button-Events mit dem Canvas bzw. den Widgets
    - zeigt Canvas, Buttons und das Log-Output-Widget an
    '''
    global GRID_SIZE, NUMBER_OF_MINES, BOARD_SPEC
    GRID_SIZE = grid_size
    NUMBER_OF_MINES = nmines
    BOARD_SPEC = (CELL_SIZE, CELL_SIZE, CELL_SIZE, CELL_SIZE, GRID_SIZE, GRID_SIZE)

    CANVAS_SIZE = (GRID_SIZE + 2) * CELL_SIZE

    canvas_config = {
        'width': CANVAS_SIZE,
        'height': CANVAS_SIZE,
        'layout': {
            'border': '1px solid black',
            'width': f'{CANVAS_SIZE}px',
            'height': f'{CANVAS_SIZE}px',
            'min_width': f'{CANVAS_SIZE}px',
            'min_height': f'{CANVAS_SIZE}px',
            'max_width': f'{CANVAS_SIZE}px',
            'max_height': f'{CANVAS_SIZE}px',
        },
    }

    canvas = Canvas(**canvas_config) 



    initialize_game(canvas)

    # Maus-Events
    canvas.on_mouse_down(lambda x, y: handle_mouse_click(x, y, canvas))

    # Flaggen-Modus-Änderungen beobachten
    flag_mode_button.observe(on_flag_mode_change, names='value')

    new_game_button = Button(description='Neues Spiel')
    new_game_button.on_click(lambda bt: new_game_clicked(bt, canvas))

    buttons_row = HBox([new_game_button, flag_mode_button])

    # ui = VBox([canvas, buttons_row, output_area])
    # display(ui)
    display(canvas, buttons_row, output_area)